# Stage 9 — does a fine-tuned reranker change the rerank-depth verdict?

**The question.** Stage 5 concluded that reranking the BGE top-50 always loses to the top-20 — but that was measured with the **off-the-shelf** cross-encoder, which Stage 8 showed is weak here (fine-tuning bought +0.107 R@1). The deeper pool demonstrably holds more answers (Stage 5: pool@20 0.9754 → pool@50 0.9951 at transformer 15/1), so the verdict should depend on how well the ranker sorts them. **Does fine-tuning flip it?**

**Why this is cheap.** Cross-encoder scores are independent per (question, chunk) pair, so the depth-20 arm is exactly the depth-50 scores restricted to the top-20 dense candidates. Every depth is scored **once** at depth 50 — no pair is scored twice — and the derived depth-20 rows double as a free reproduction check against `stage8/final`.

**Cost (T4).** Default 2 configs ≈ **1.5 h**; `--configs all` ≈ 4.1 h (will not fit one free session — rely on the checkpoint).

**Before running:** Runtime → Change runtime type → **GPU (T4)**, then Run all. Needs the fine-tuned reranker at `artifacts/models/bge_reranker_ft/final` and the Stage 6 bench cache.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
# Set RAG_PROJECT_DIR to the project root before running this cell.
PROJECT_DIR = os.environ.get('RAG_PROJECT_DIR')
if not PROJECT_DIR:
    raise RuntimeError('Set RAG_PROJECT_DIR to the project directory before running this notebook.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print(C.summary())

## Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## Smoke run first (~8 min) — verifies the pipeline end to end

Scores only 30 questions. The numbers are **not** results and the Stage 8 check is skipped; outputs are written with a `smoke_` prefix so they can never be archived by mistake. If this finishes cleanly, the real run will too.

In [ ]:
!python scripts/22_rerank_depth.py --max-questions 30

## The real run (~1.5 h)

Checkpointed per config — if the session dies, re-run this cell and it resumes. Watch for `check OK` on the depth-20 arms: the depth-50 numbers only mean something once the depth-20 ones reproduce `stage8/final`.

In [ ]:
!python scripts/22_rerank_depth.py

## Review

In [ ]:
from IPython.display import Image, Markdown, display
import pathlib, os
latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
display(Markdown((latest / 'stage9_depth_summary.md').read_text(encoding='utf-8')))
display(Image(str(latest / 'stage9_depth_delta.png')))

## Archive (only if the check passed)

In [ ]:
!python scripts/save_stage_results.py --stage stage9

## How to read the verdict

The console prints, per config, `ft` deep-minus-shallow ΔR@1 against 2 SE:

- **DEEPER WINS** — the Stage 5 conclusion flips once the ranker is strong enough. A genuinely interesting result: a claim that held for a weak reranker does not generalise to a good one.
- **NO DIFFERENCE** — the deeper pool buys nothing measurable; keep depth 20 and save 2.5× the reranking cost. Stage 5's advice survives for a different reason.
- **DEEPER LOSES** — Stage 5 holds even for the fine-tuned model; the extra candidates are genuinely harmful.

All three are honest, reportable outcomes. Compare the `ots` column too: if off-the-shelf still loses while fine-tuned wins, that contrast **is** the finding.

Whatever happens, the chunking headline (size dominates, method ties) is untouched — this only tunes the reranking stage.